Imports and setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.config import TARGET_COLUMN, PROCESSED_DATA

import pandas as pd

from src.preprocessing_utils import (
    # chronological_train_test_split,
    create_sequences,
    load_feature_engineered_dataset,
    parse_datetime_index,
    scale_train_test_data,
    split_features_and_target,
)

from src.model_utils import (
    create_lstm_model,
    create_model_comparison_dataframe,
    evaluate_regression_model,
    get_early_stopping_callback,
    plot_actual_vs_predicted,
    plot_training_history,
    save_model_comparison_dataframe,
    train_linear_regression_model,
    train_random_forest_model,
    save_model
)

pd.set_option("display.max_columns", None)

### Model Development and Evaluation

This notebook develops baseline machine learning models and a deep learning model for appliance energy prediction.  
The models are evaluated using Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE).

Loading the preprocessed datasets 

In [ ]:
train_path = PROCESSED_DATA / "train_feature_engineered_data.csv"
test_path = PROCESSED_DATA / "test_feature_engineered_data.csv"

train_df = load_feature_engineered_dataset(train_path)
test_df = load_feature_engineered_dataset(test_path)

train_df = parse_datetime_index(train_df, "date")
test_df = parse_datetime_index(test_df, "date")

In [ ]:
# checking dataset shapes
print("Training set shape:", train_df.shape)
print("Testing set shape:", test_df.shape)

Split features and targets

In [ ]:
x_train_df, y_train_series = split_features_and_target(train_df, TARGET_COLUMN)
x_test_df, y_test_series = split_features_and_target(test_df, TARGET_COLUMN)

print("X_train shape:", x_train_df.shape)
print("X_test shape:", x_test_df.shape)
print("y_train shape:", y_train_series.shape)
print("y_test shape:", y_test_series.shape)

Scaling features for baseline and LSTM 

In [ ]:
x_train_scaled, x_test_scaled, fitted_scaler = scale_train_test_data(
    x_train=x_train_df,
    x_test=x_test_df,
    scaler_type="standard",
)

print("Scaled X_train shape:", x_train_scaled.shape)
print("Scaled X_test shape:", x_test_scaled.shape)

### Training Linear Regression Model

In [ ]:
linear_model = train_linear_regression_model(
    x_train=x_train_scaled,
    y_train=y_train_series.to_numpy(),
)

linear_predictions = linear_model.predict(x_test_scaled)

linear_metrics = evaluate_regression_model(
    y_true=y_test_series.to_numpy(),
    y_pred=linear_predictions,
)

linear_metrics

In [ ]:
# saving the Linear Regression model
save_model(
    model=linear_model,
    file_name="linear_regression_model.pkl",
)

### Training Random Forest

In [ ]:
random_forest_model = train_random_forest_model(
    x_train=x_train_scaled,
    y_train=y_train_series.to_numpy(),
    n_estimators=200,
    random_state=42,
)

random_forest_predictions = random_forest_model.predict(x_test_scaled)

random_forest_metrics = evaluate_regression_model(
    y_true=y_test_series.to_numpy(),
    y_pred=random_forest_predictions,
)

random_forest_metrics

In [ ]:
# saving the Random Forest model
save_model(
    model=random_forest_model,
    file_name="random_forest_model_model.pkl",
)

Plotting the baseline model predictions (Linear regressiond and Random forest)

In [ ]:
plot_actual_vs_predicted(
    y_true=y_test_series.to_numpy()[:300],
    y_pred=linear_predictions[:300],
    title="Linear Regression: Actual vs Predicted",
    file_name="linear_regression_actual_vs_predicted.png"
)

plot_actual_vs_predicted(
    y_true=y_test_series.to_numpy()[:300],
    y_pred=random_forest_predictions[:300],
    title="Random Forest: Actual vs Predicted",
    file_name="random_forest_actual_vs_predicted.png"
)

### LSTM model

Creating sequences for LSTM model

In [ ]:
sequence_length = 6

x_train_seq, y_train_seq = create_sequences(
    x_data=x_train_scaled,
    y_data=y_train_series.to_numpy(),
    sequence_length=sequence_length,
)

x_test_seq, y_test_seq = create_sequences(
    x_data=x_test_scaled,
    y_data=y_test_series.to_numpy(),
    sequence_length=sequence_length,
)

print("x_train_seq shape:", x_train_seq.shape)
print("y_train_seq shape:", y_train_seq.shape)
print("x_test_seq shape:", x_test_seq.shape)
print("y_test_seq shape:", y_test_seq.shape)

Builing the model

In [ ]:
lstm_model = create_lstm_model(
    input_shape=(x_train_seq.shape[1], x_train_seq.shape[2]),
    lstm_units=64,
    dropout_rate=0.2,
)

lstm_model.summary()

Training the model

In [ ]:
early_stopping = get_early_stopping_callback(patience=5)

history = lstm_model.fit(
    x_train_seq,
    y_train_seq,
    validation_split=0.1,
    epochs=30,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1,
)

In [ ]:
# saving the LSTM model
save_model(
    model=lstm_model,
    file_name="lstm_model.pkl",
)

In [ ]:
# plotting training history
plot_training_history(history)

Evaluating the LSTM model

In [ ]:
lstm_predictions = lstm_model.predict(x_test_seq).flatten()

lstm_metrics = evaluate_regression_model(
    y_true=y_test_seq,
    y_pred=lstm_predictions,
)

lstm_metrics

In [ ]:
# plotting LSTM model predictions
plot_actual_vs_predicted(
    y_true=y_test_seq[:300],
    y_pred=lstm_predictions[:300],
    title="LSTM: Actual vs Predicted",
    file_name="lstm_actual_vs_predicted.png"
)

### Model Comparison 

In [ ]:
comparison_df = create_model_comparison_dataframe(
    linear_metrics=linear_metrics,
    random_forest_metrics=random_forest_metrics,
    lstm_metrics=lstm_metrics,
)

comparison_df

In [ ]:
# saving comparison table
save_model_comparison_dataframe(
    comparison_df=comparison_df,
    file_name="model_comparison_results.csv"
)